# Build model — NEW aggregator WITH the experian placeholder change

Identical to `Build_Model_New.ipynb` (same features, same applicants, same
pipeline, same data_split) with ONE difference: the **experian** trade
features come from

    payment_processing_research_data/new_normalized_and_processed/experian_<role>/processed/

(built by `NewProcessing_Experian_{Train,Test}.ipynb` with the
`payment_processor_change` branches — placeholder removed, dashes kept)
instead of the experiment-era `samples/experian_<role>/processed_new/`
(built BEFORE the placeholder change). Equifax and transunion still use
`processed_new` — for them the branch pipeline is behavior-identical, so
re-processing would have produced the same files.

Output -> `payment_processing_research_data/models/model_new_with_change/`.
Compare against `models/model_new/` and `models/model_old/` with the
Evaluate notebooks. Run in the SAME model-engine kernel as
`Build_Model_New.ipynb` (the trade features are already materialized; this
notebook does not need the branch installs).

In [1]:
import os, glob, json, importlib
import model_configs
importlib.reload(model_configs)
from configs import DATA_DIR
from model_engine.model_builder.build_model import build_model

VARIANT = 'new_with_change'
out_dir = os.path.join(model_configs.MODELS_DIR, f'model_{VARIANT}')
os.makedirs(out_dir, exist_ok=True)
NEW_DIR = os.path.join(DATA_DIR, 'new_normalized_and_processed')
print('variant:', VARIANT, '| output ->', out_dir)

/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



variant: new_with_change | output -> /home/jag/payment-processor-research/payment_processing_research_data/models/model_new_with_change


In [2]:
# Start from the standard NEW asset, then swap ONLY the experian trade files
# to the new_normalized_and_processed location.
asset = model_configs.build_asset('new')

trade_files = asset['data']['trade']['data']
keep        = [f for f in trade_files if f'{os.sep}experian_' not in f]
dropped     = len(trade_files) - len(keep)

new_exp = sorted(glob.glob(os.path.join(NEW_DIR, 'experian_*', 'processed', 'part-*.parquet')))
assert new_exp, f'no files under {NEW_DIR}/experian_*/processed -- run NewProcessing_Experian_{{Train,Test}}.ipynb first'
for role in ['train', 'test']:
    n = len([f for f in new_exp if f'experian_{role}' in f])
    assert n > 0, f'missing experian_{role} processed files'
    print(f'experian_{role}: {n} new files')

asset['data']['trade']['data'] = keep + new_exp
print(f'trade files: dropped {dropped} experiment-era experian, added {len(new_exp)} '
      f'placeholder-change experian, total {len(asset["data"]["trade"]["data"])}')

experian_train: 50 new files
experian_test: 50 new files
trade files: dropped 2 experiment-era experian, added 100 placeholder-change experian, total 104


In [4]:
new_exp

['/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-000.parquet',
 '/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-001.parquet',
 '/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-002.parquet',
 '/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-003.parquet',
 '/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-004.parquet',
 '/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-005.parquet',
 '/home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_test/processed/part-006.pa

In [3]:
asset

{'data': {'app': {'asset': {'info': {'key': 'ZEST_KEY', 'app_date': 'appDate'},
    'table_name': 'app',
    'table_type': 'one_to_one',
    'feature_engineering': 'one_to_one'},
   'data': ['/home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_train/app.parquet',
    '/home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_test/app.parquet',
    '/home/jag/payment-processor-research/payment_processing_research_data/samples/experian_train/app.parquet',
    '/home/jag/payment-processor-research/payment_processing_research_data/samples/experian_test/app.parquet',
    '/home/jag/payment-processor-research/payment_processing_research_data/samples/transunion_train/app.parquet',
    '/home/jag/payment-processor-research/payment_processing_research_data/samples/transunion_test/app.parquet'],
   'io_params': {'drop_duplicates': True,
    'keep_features': None,
    'memory_efficient': True}},
  'trade': {'asset': {'fe_version': 2,
 

In [5]:
# sanity: every table has files before we train
for tbl in ['app', 'trade', 'target']:
    n = len(asset['data'][tbl]['data'])
    print(f'{tbl:7}: {n} files')
    assert n > 0, f'no {tbl} files'
print('trade keep_features:', 'ALL' if asset['data']['trade']['io_params']['keep_features'] is None
      else len(asset['data']['trade']['io_params']['keep_features']))
print('data_split:', asset['config']['data_split'])

json.dump(asset, open(os.path.join(out_dir, 'asset.json'), 'w'), indent=2)
print('wrote', os.path.join(out_dir, 'asset.json'))

app    : 6 files
trade  : 104 files
target : 6 files
trade keep_features: 1143
data_split: {'train': {'start_date': '2019-04-01', 'end_date': '2019-07-01'}, 'test': {'start_date': '2019-07-01', 'end_date': '2020-04-01'}}
wrote /home/jag/payment-processor-research/payment_processing_research_data/models/model_new_with_change/asset.json


In [6]:
import logging, traceback

log_path = os.path.join(out_dir, f'build_model_{VARIANT}.log')
fh = logging.FileHandler(log_path, mode='w')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
root = logging.getLogger(); root.addHandler(fh); root.setLevel(logging.INFO)

try:
    build_model(asset, out_dir)
    root.info('BUILD SUCCEEDED -> %s', out_dir)
    print('\nDONE -> model artifacts in', out_dir)
    print(sorted(os.listdir(out_dir)))
except Exception as e:
    root.error('BUILD FAILED: %s\n%s', e, traceback.format_exc())
    print('BUILD FAILED -- see log:', log_path)
    raise
finally:
    root.removeHandler(fh); fh.close()
    print('log saved ->', log_path)

parsing model-builder asset


Configuring model builder


INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_data>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <monotonic_constraints_list>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <add_default_monotonic_constraints>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <add_default_monotonic_constraints>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <fe_version>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <data_split>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <train_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <train_sample_weight>, thus it will be omitt

building model


INFO:zaml.artifact_engine.logger:Finished <versions>, total time spent: 0:00:06.229664
INFO:zaml.artifact_engine.logger:Executing MonotonicConstraintsListParser <parsed_monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Finished <parsed_monotonic_constraints_list>, total time spent: 0:00:01.130536
INFO:zaml.artifact_engine.logger:Executing SplitterArtifact <splitter>...
INFO:zaml.artifact_engine.logger:Finished <splitter>, total time spent: 0:00:00.000453
INFO:zaml.artifact_engine.logger:Executing DataArtifact <data>...
INFO:zaml.artifact_engine.logger:Finished <data>, total time spent: 0:00:00.000227
INFO:zaml.artifact_engine.logger:Executing ExclusionListParser <parsed_exclusion_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <parsed_exclusion_list>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Finished <parsed_exclusion_list>, total time spent: 0:00:00.000420
INFO:zaml.artifact_engine.logger:Executing Bi

...

-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 1143
Time spent: 20.692s
-------------------------
Name: app
Transformer type: None
Number of features: 52
Time spent: 0.000s
-------------------------
Name: trade
Transformer type: None
Number of features: 1143
Time spent: 0.000s
-------------------------
Name: app FE
Transformer type: OneToOneEngine
Number of features: 0
Time spent: 0.461s
-------------------------
Name: trade FE
Transformer type: EndtoEndFeatureEngine
Number of features: 1143
Time spent: 2.601s
-------------------------
Name: Merge data
Transformer type: Concat
Number of features: 1143
Time spent: 12.009s
-------------------------
Name: LevelSelection
Transformer type: LevelSelection
Number of features: 1143
Time spent: 2.764s


INFO:zaml.artifact_engine.logger:Finished <test_fe_data>, total time spent: 0:00:38.751215
INFO:zaml.artifact_engine.logger:Executing StaticAssetArtifact <static_asset>...
INFO:zaml.artifact_engine.logger:Finished <static_asset>, total time spent: 0:00:00.196841
INFO:zaml.artifact_engine.logger:Executing TrainHistoryArtifact <train_history>...
INFO:zaml.artifact_engine.logger:Finished <train_history>, total time spent: 0:00:00.000267
INFO:zaml.artifact_engine.logger:Executing BestModelParamsArtifact <best_model_params>...
INFO:zaml.artifact_engine.logger:Finished <best_model_params>, total time spent: 0:00:00.000227
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <train_scores>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 1143
Time spent: 20.861s


INFO:zaml.artifact_engine.logger:Finished <train_scores>, total time spent: 0:00:08.033713
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <train_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <train_submodel_scores>, total time spent: 0:00:00.000283
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <test_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_scores>, total time spent: 0:00:08.030662
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <test_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_submodel_scores>, total time spent: 0:00:00.000337
INFO:zaml.artifact_engine.logger:Executing CalibrationObjectArtifact <calibration_object>...
INFO:zaml.artifact_engine.logger:Finished <calibration_object>, total time spent: 0:00:11.027590
INFO:zaml.artifact_engine.logger:Executing FeatureDefinition <feature_definition>...
INFO:zaml.artifact_engine.logger:Finished <feature_definition>, total time spent: 0:0


DONE -> model artifacts in /home/jag/payment-processor-research/payment_processing_research_data/models/model_new_with_change
['artifact_manifest.json', 'asset.json', 'best_model_params.json', 'build_model_new_with_change.log', 'calibration_object.obj', 'feature_definition.parquet', 'feature_importance.parquet', 'fit_time_info.json', 'keep_features.json', 'key_factors_mapping.json', 'model.obj', 'model_strategy.json', 'mrm_pipeline.obj', 'parsed_monotonic_constraints_list.json', 'pipeline.obj', 'score_recalibration_mapping.json', 'splitter.obj', 'static_asset.json', 'test_app.parquet', 'test_auc.json', 'test_data_summary.json', 'test_fe_data.parquet', 'test_ks.json', 'test_scores.parquet', 'test_target.parquet', 'test_zest_scores.parquet', 'top_features.parquet', 'train_app.parquet', 'train_auc.json', 'train_data_summary.json', 'train_fe_data.parquet', 'train_history.json', 'train_ks.json', 'train_scores.parquet', 'train_target.parquet', 'train_zest_scores.parquet', 'value_based_key_f